# conv-leakyrelu-block-discriminator — ex1: build a 32x32 to 16x16 discriminator downsampling block

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-leakyrelu-block-discriminator`. Running the final beacon cell reports progress against the `GAN: Conv+LeakyReLU discriminator block` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Conv+LeakyReLU discriminator block` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-leakyrelu-block-discriminator`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-leakyrelu-block-discriminator"
DD_SUBTOPIC = "GAN: Conv+LeakyReLU discriminator block"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv + BN + LeakyReLU discriminator block — quick refresher

The repeated unit of a DCGAN discriminator. Three layers in `nn.Sequential`, downsampling by 2 every block:

```python
nn.Sequential(
    nn.Conv2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(out_c),
    nn.LeakyReLU(0.2, inplace=True),
)
```

**Stride 2, kernel 4, padding 1 halves spatial size.** Output `H_out = (H_in + 2*padding - kernel) // stride + 1 = H_in // 2`. So `32 -> 16 -> 8 -> 4 -> 1` for a 4-block stack on 32×32 input.

**LeakyReLU(0.2) — the discriminator default.** Slope 0.2 on the negative side keeps gradient flowing for inputs the discriminator currently thinks are fake. Plain ReLU would zero those gradients — the discriminator would stop learning from negative examples.

**First block usually skips BN.** ARENA's implementation has the FIRST conv block of the discriminator omit BatchNorm — adding it on the raw RGB input scales the image stats away. All INTERMEDIATE blocks include BN.

### Exercise 1 — build a 32x32 to 16x16 discriminator downsampling block

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `nn.Sequential` to wire `nn.Conv2d(stride=2, kernel=4, padding=1, bias=False) -> nn.BatchNorm2d -> nn.LeakyReLU(0.2)` into a single channel-doubling, spatial-halving discriminator block.
> Keywords: gan, discriminator, conv2d, leakyrelu, sequential
> ```

**KCs targeted:** `sequential-three-layer-block`, `conv-stride-halves-spatial`

Implement `ex1_build_discriminator_block(in_channels, out_channels)`. The repeated unit of a DCGAN discriminator:

1. Construct an `nn.Sequential` containing three layers IN ORDER:
   - `nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)`
   - `nn.BatchNorm2d(out_channels)`
   - `nn.LeakyReLU(negative_slope=0.2, inplace=True)`
2. `bias=False` on the Conv because BatchNorm immediately follows.
3. Stride 2 + kernel 4 + padding 1 HALVES the spatial size — a 32×32 input becomes 16×16 output.
4. Return the Sequential.

(Note: in a real DCGAN, the FIRST block omits BatchNorm because input is raw image stats. This drill builds an INTERMEDIATE block, so BN is included.)

Input: `in_channels`, `out_channels` — ints.
Output: `nn.Sequential` module.

The visualization renders four output channel slices as a 2×2 grid of 16×16 feature maps.

In [ ]:
def ex1_build_discriminator_block(in_channels: int, out_channels: int) -> t.nn.Sequential:
    import torch.nn as nn
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.LeakyReLU(negative_slope=0.2, inplace=True),
    )


<details><summary>Solution</summary>

```python
def ex1_build_discriminator_block(in_channels: int, out_channels: int) -> t.nn.Sequential:
    import torch.nn as nn
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.LeakyReLU(negative_slope=0.2, inplace=True),
    )
```

**Why kernel=4, stride=2, padding=1.** The discriminator mirror of the generator block. Output size for Conv2d is `H_out = (H_in + 2 * padding - kernel) // stride + 1 = (H_in - 2) // 2 + 1 = H_in // 2`. Exact halving.

**Why LeakyReLU(0.2), not plain ReLU.** Plain ReLU zeroes out all negative inputs — including the discriminator's score for samples it currently thinks are fake. Gradient is zero on those, so the discriminator stops learning. LeakyReLU(0.2) gives a 0.2× gradient on negatives — still flowing, just attenuated. The 0.2 value comes straight from the DCGAN paper.

**Real DCGAN: first block has NO BN.** The very first block downsamples raw image input (e.g. 64x64 RGB → 32x32 features). BatchNorm on raw pixel stats washes out the image. ARENA's reference implementation passes `skip_first_bn=True` to handle this — but here we build a clean intermediate block.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()